In [61]:
import json
import pandas as pd

with open("games.json") as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df = df.reset_index(drop=True)

The first thing for pre-processing is cleaning up the estimated owners feature, which will be the target variable for my model. The dataset returns these in ranges of values, as the actual numerical data is inaccesible. I will be using regression models to compare the potency of my second layer of my models, and such I will need a target column of singular numbers, which will be done by getting the midpoint of the range of each samples estimated owners. I then apply a log transformation due to the massive gaps between the high value estimated owner categories.

In [62]:
df['estimated_owners_raw'] = df['estimated_owners']

def parse_owners(val):
    try:
        low, high = val.split(' - ')
        return (int(low) + int(high)) / 2
    except:
        return None

df['owners_numeric'] = df['estimated_owners_raw'].apply(parse_owners)

import numpy as np
df['log_owners'] = np.log1p(df['owners_numeric'])


I also need to convert tags to just a list of the names, as the data put tags into dicts with numeric values attached that don't have any value to my models. Tag similarity is important to my final product, and for that I only want the names.

In [ ]:
def clean_tags(x):
    if isinstance(x, dict):
        return list(x.keys())
    elif isinstance(x, list):
        return x
    else:
        return []

df['tags_clean'] = df['tags'].apply(clean_tags)

df['num_tags'] = df['tags_clean'].apply(len)

Convert written release dates to datetime

In [64]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year


Create New DF with dropped rows missing important values

In [ ]:
total = len(df)

missing_owners = df['estimated_owners_raw'].isnull().sum()
missing_desc = df['detailed_description'].isnull().sum()
empty_desc = (df['detailed_description'].str.strip() == '').sum()
empty_tags = (df['tags_clean'].apply(len) == 0).sum()

print(f"Total rows: {total}")
print(f"Missing owners: {missing_owners}")
print(f"Missing descriptions: {missing_desc}")
print(f"Empty descriptions: {empty_desc}")
print(f"Empty tags: {empty_tags}")

df['description'] = df['detailed_description'].fillna('').astype(str)

df = df[
    (df['description'].str.strip() != '') &
    (df['estimated_owners_raw'].notnull())
]


Total rows: 122611
Missing owners: 0
Missing descriptions: 0
Empty descriptions: 8420
Empty tags: 39265


To properly create my target value, I need to order the defferent ranges of estimated owners and observe the distribution, then decide how I will classify each instance, likely in four categories, Low Sale, Medium Sale, High Sale, and Extreme Success.

In [66]:
buckets = [
    "0 - 20000",
    "20000 - 50000",
    "50000 - 100000",
    "100000 - 200000",
    "200000 - 500000",
    "500000 - 1000000",
    "1000000 - 2000000",
    "2000000 - 5000000",
    "5000000 - 10000000",
    "10000000 - 20000000",
    "20000000 - 50000000",
    "50000000 - 100000000",
    "100000000 - 200000000"
]

counts = df['estimated_owners_raw'].value_counts().reindex(buckets)
print(counts)

hit_threshold = [
    "50000 - 100000",
    "100000 - 200000",
    "200000 - 500000",
    "500000 - 1000000",
    "1000000 - 2000000",
    "2000000 - 5000000",
    "5000000 - 10000000",
    "10000000 - 20000000",
    "20000000 - 50000000",
    "50000000 - 100000000",
    "100000000 - 200000000"
]

df['hit'] = df['estimated_owners_raw'].isin(hit_threshold).astype(int)

df['hit'].value_counts()

estimated_owners_raw
0 - 20000                75321
20000 - 50000            11381
50000 - 100000            5345
100000 - 200000           3449
200000 - 500000           2847
500000 - 1000000          1154
1000000 - 2000000          728
2000000 - 5000000          401
5000000 - 10000000         124
10000000 - 20000000         51
20000000 - 50000000         30
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64


hit
0    100049
1     14142
Name: count, dtype: int64